<a href="https://colab.research.google.com/github/Kaniz-Ankan/XAI-Implementation-on-lungimg/blob/main/lung_image_xai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Explainable AI for Lung Image Classification
# Techniques used: Grad-CAM, LIME, SHAP

In [ ]:
!pip install tensorflow keras lime scikit-learn matplotlib seaborn opencv-python shap


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
import numpy as np, matplotlib.pyplot as plt, cv2, os
from lime import lime_image
from skimage.segmentation import mark_boundaries


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    '/content/drive/MyDrive/MACHINE LEARNING/Dataset/lungDsImg',
    image_size=(224,224),
    batch_size=32
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    '/content/drive/MyDrive/MACHINE LEARNING/Dataset/lungDsImg',
    image_size=(224,224),
    batch_size=32
)

class_names = train_ds.class_names
print("Classes:", class_names)


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)


In [ ]:
base_model = keras.applications.MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(3, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()


In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=50)


In [ ]:
image_batch, label_batch = next(iter(val_ds))
img = image_batch[0]
plt.imshow(img.numpy().astype("uint8"))
plt.show()

pred = model.predict(tf.expand_dims(img,0))
print("Predicted:", class_names[np.argmax(pred)])


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import cv2

# --- Grad-CAM helper functions ---
def make_gradcam_heatmap(img_array, model, base_model, last_conv_layer_name):
    # Access the internal conv layer from base_model
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[base_model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        class_idx = tf.argmax(predictions[0])
        loss = predictions[:, class_idx]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_mean(tf.multiply(pooled_grads, conv_outputs), axis=-1)
    heatmap = np.maximum(heatmap, 0)
    if np.max(heatmap) != 0:
        heatmap /= np.max(heatmap)
    return heatmap.numpy()


def overlay_heatmap(heatmap, img, alpha=0.4):
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    superimposed = cv2.addWeighted(heatmap, alpha, img.astype('uint8'), 1-alpha, 0)
    return superimposed


# --- Get last Conv2D layer from base_model ---
last_conv_layer = None
for layer in reversed(base_model.layers):
    if isinstance(layer, tf.keras.layers.Conv2D):
        last_conv_layer = layer.name
        break

if last_conv_layer is None:
    raise ValueError("Could not find a Conv2D layer in base_model.")
print("Using last conv layer:", last_conv_layer)

# --- Select one image from validation set ---
image_batch, label_batch = next(iter(val_ds))
img = image_batch[0]
plt.imshow(img.numpy().astype("uint8"))
plt.title("Original Image")
plt.axis('off')
plt.show()

# --- Generate Grad



In [ ]:
from lime import lime_image
from skimage.segmentation import mark_boundaries
import numpy as np
import matplotlib.pyplot as plt

explainer = lime_image.LimeImageExplainer()

# Convert to float for LIME
img_np = np.array(img).astype('double')

# Explain the prediction
explanation = explainer.explain_instance(
    img_np,
    classifier_fn=lambda x: model.predict(x),
    top_labels=3,
    hide_color=0,
    num_samples=1000
)

# Get the mask for the predicted class
pred_label = np.argmax(model.predict(tf.expand_dims(img,0)))
temp, mask = explanation.get_image_and_mask(
    label=pred_label,
    positive_only=True,
    num_features=5,
    hide_rest=False
)

# Display the explanation
plt.imshow(mark_boundaries(temp/255.0, mask))
plt.title(f"LIME Explanation (Class: {class_names[pred_label]})")
plt.axis('off')
plt.show()



In [ ]:
import shap
import numpy as np

# Convert TensorFlow tensors to NumPy arrays
background = image_batch[:10].numpy()
test_image = image_batch[:1].numpy()

# Create SHAP GradientExplainer
e = shap.GradientExplainer(model, background)

# Compute SHAP values for one test image
shap_values = e.shap_values(test_image)

# Plot the SHAP explanation
shap.image_plot(shap_values, test_image)

